# Training Series: Introduction to Discontinuous Galerkin Methods for Flow Problems <br> - Implementation of a DG solver for incompressible single-phase flows

In this tutorial/exercise you will implement a DG-solver for incompressible single-phase flows.

## Problem statement: transient incompressible single-phase flow

The flow within the computational domain $\Omega \in R^2$ is described by the unsteady Navier-Stokes equations 

>$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \vec{u} \cdot \nabla \vec{u}\right) = -\nabla p + \mu_f \Delta \vec{u} = \rho_f \vec{f} \quad \textrm{in}\ \Omega $$

>$$ \rho_f\left(\frac{\partial \vec{u}}{\partial t}+ \nabla \cdot \left( \vec{u} \otimes \vec{u} \right) \right) = \nabla \cdot \left( -p\mathbf{I} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \right) = \rho_f \vec{f} \quad \textrm{in}\ \Omega $$

and the continuity equation
>$$ \nabla \cdot \vec{u} = 0 \quad \forall\ t \in (0,T)\quad \textrm{in}\ \Omega $$

In the equations above 
 - $\vec{u}$ is the velocity vector 
 - $p$ the pressure
 - and $\vec{f}$ some volume force, e.g. gravity. 
 - The fluid density is denoted by $\rho_f$
 - $\mu_f=\rho_f \cdot \nu_f$ is the dynamic viscosity of the fluid.
 
At the boundary $\partial \Omega = \partial \Omega_\textrm{D} \cup \partial \Omega_\textrm{N}$ the corresponding boundary conditions read 

>$$ \vec{u} = \vec{u}_\textrm{D} \quad \textrm{on}\ \partial \Omega_\textrm{D} $$

>$$ -p \vec{n}_{\partial \Omega} + \mu_f \left( \nabla \vec{u} + \nabla \vec{u}^{\textrm{T}} \right) \vec{n}_{\partial \Omega} = - p_\textrm{ext} \vec{n}_{\partial \Omega} \quad \textrm{on}\ \partial \Omega_\textrm{N} $$


## Implementation of the DG solver

In [2]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

Unhandled exception: System.TypeLoadException: Method 'get_MPI_Comm' in type 'ilPSP.LinSolvers.BlockMsrMatrix' from assembly 'BoSSS.Platform, Version=1.0.0.0, Culture=neutral, PublicKeyToken=null' does not have an implementation.
   at BoSSS.Application.BoSSSpad.BoSSSshell.CallRandomStuff()
   at BoSSS.Application.BoSSSpad.BoSSSshell.Init() in D:\BoSSS-experimental\public\src\L4-application\BoSSSpad\BoSSSshell.cs:line 78
   at Submission#5.<<Initialize>>d__0.MoveNext()
--- End of stack trace from previous location ---
   at Microsoft.CodeAnalysis.Scripting.ScriptExecutionState.RunSubmissionsAsync[TResult](ImmutableArray`1 precedingExecutors, Func`2 currentExecutor, StrongBox`1 exceptionHolderOpt, Func`2 catchExceptionOpt, CancellationToken cancellationToken)

In [3]:
using BoSSS.Solution.NSECommon;
using BoSSS.Application.IncompressibleNSE;
using System.Runtime.Serialization;

Define our own solver class as a derivative from `IncompressibleNSEMain`

In [4]:
public class NavierStokesDGSolver : IncompressibleNSEMain {

    static void Main(string[] args) {
        _Main(args, false, delegate () {
            var p = new NavierStokesDGSolver();
            return p;
        });
    }

    /// <summary>
    /// Declaration of the spatial operator
    /// </summary>
    protected override SpatialOperator GetOperatorInstance(int D) {

        // instantiate boundary condition mapping
        // ======================================
        boundaryCondMap = new IncompressibleBoundaryCondMap(this.GridData, this.Control.BoundaryValues, PhysicsMode.Incompressible);

        // instantiate operator
        // ====================
        string[] CodName = (new[] { "ResidualMomentumX", "ResidualMomentumY", "ResidualMomentumZ" }).GetSubVector(0, D).Cat("ResidualConti");

        var op = new SpatialOperator(
            __DomainVar: VariableNames.VelocityVector(D).Cat(VariableNames.Pressure),
            __ParameterVar: VariableNames.GravityVector(D),
            __CoDomainVar: CodName,
            QuadOrderFunc: QuadOrderFunc.NonLinear(2));

        op.LinearizationHint = LinearizationHint.GetJacobiOperator;

        // Temporal Operator
        // =================

        var TempOp = new ConstantTemporalOperator(op, 0.0); // init with entire diagonal set to 0.0
        op.TemporalOperator = TempOp;

        for (int d = 0; d < D; d++)
            TempOp.SetDiagonal(CodName[d], Control.Density); // set momentum equation entries to density

        // Pressure Reference
        // ==================

        // if there is no Dirichlet boundary condition,
        // the mean value of the pressure is free:
        op.FreeMeanValue[VariableNames.Pressure] = !boundaryCondMap.DirichletPressureBoundary;

        // Momentum Equation
        // =================

        // convective part:
        {
            for (int d = 0; d < D; d++) {
                var comps = op.EquationComponents[CodName[d]];
                var ConvBulk = new LocalLaxFriedrichsConvection(D, boundaryCondMap, d, Control.Density, null);
                comps.Add(ConvBulk); // bulk component
            }
        }

        // pressure part:
        {
            for (int d = 0; d < D; d++) {
                var comps = op.EquationComponents[CodName[d]];
                var pres = new PressureGradientLin_d(d, boundaryCondMap);
                comps.Add(pres); // bulk component
            }
        }

        // viscous part:
        {
            for (int d = 0; d < D; d++) {
                var comps = op.EquationComponents[CodName[d]];

                double penalty_bulk = this.Control.PenaltySafety;

                var Visc = new SipViscosity_GradU(penalty_bulk, d, D, boundaryCondMap,
                    ViscosityOption.ConstantViscosity,
                    constantViscosityValue: Control.Viscosity);
                comps.Add(Visc); // bulk component GradUTerm
            }
        }


        // Continuity equation
        // ===================
        {
            for (int d = 0; d < D; d++) {
                var src = new Divergence_DerivativeSource(d, D);
                var flx = new Divergence_DerivativeSource_Flux(d, boundaryCondMap);
                op.EquationComponents[CodName[D]].Add(src);
                op.EquationComponents[CodName[D]].Add(flx);
            }
        }

        // Gravity parameter
        // =================

        op.ParameterFactories.Add(delegate (IReadOnlyDictionary<string, DGField> DomainVarFields) {
            return D.ForLoop(d => (VariableNames.Gravity_d(d), this.Gravity[d] as DGField));
        });


        // commit & return
        // ===============
        op.Commit();
        return op;

    }

    protected override double RunSolverOneStep(int TimestepNo, double phystime, double dt) {

        dt = Control.dtFixed;

        Console.WriteLine(" ");
        Console.WriteLine($"Starting time-step #{TimestepNo}, dt = {dt}...");

        Solve(phystime, dt);

        Console.WriteLine("done.");
        return dt;
    }

}


(3,17): warning CS7022: The entry point of the program is global code; ignoring 'NavierStokesDGSolver.Main(string[])' entry point.



Define a corresponding control class

In [5]:
[Serializable]
[DataContract]
public class NavierStokesdDGSolverControl : IncompressibleControl {
   
    /// <summary>
    /// required for BoSSSpad workflow management
    /// </summary>
    public override Type GetSolverType() {
        return typeof(NavierStokesDGSolver);
    }
}

## Setup test case

In [ ]:
// // create grid
// int gridResolution = 2;

// double Length = 10.0;
// double Height = 2.0;

// double[] _xNodes = GenericBlas.Linspace(0, Length, gridResolution * 5 + 1);
// double[] _yNodes = GenericBlas.Linspace(-Height/2.0, Height/2.0, gridResolution + 1);

// GridCommons grid = Grid2D.Cartesian2DGrid(_xNodes, _yNodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

// grid.DefineEdgeTags(delegate (Vector _X) {
//     double x = _X[0];
//     double y = _X[1];

//     if(Math.Abs(y - (-Height/2.0)) < 1.0e-6)
//         // bottom
//         return IncompressibleBcType.Wall.ToString() + "_bottom";

//     if(Math.Abs(y - (+Height/2.0)) < 1.0e-6)
//         // top
//         return IncompressibleBcType.Wall.ToString()+ "_top";


//     if(Math.Abs(x - (0.0)) < 1.0e-6)
//         // left
//         return IncompressibleBcType.Velocity_Inlet.ToString();

//     if(Math.Abs(x - (Length)) < 1.0e-6)
//         // right
//         return IncompressibleBcType.Pressure_Outlet.ToString();

//     throw new ArgumentOutOfRangeException();
// });

Grid Edge Tags changed.


In [6]:
NavierStokesdDGSolverControl C = new NavierStokesdDGSolverControl();

// database and saving options
C.DbPath = null;
C.savetodb = false;
C.ProjectName = "ChannelFlow";

// Physical parameters
C.Density = 1.0;
C.Viscosity = 1.0 / 10.0;

// DG degree
C.SetDGdegree(3);

// computational grid
//C.SetGrid(grid);
C.GridFunc = delegate {
    int gridResolution = 2;

    double Length = 10.0;
    double Height = 2.0;

    double[] _xNodes = GenericBlas.Linspace(0, Length, gridResolution * 5 + 1);
    double[] _yNodes = GenericBlas.Linspace(-Height/2.0, Height/2.0, gridResolution + 1);

    var grd = Grid2D.Cartesian2DGrid(_xNodes, _yNodes, BoSSS.Foundation.Grid.RefElements.CellType.Square_Linear);

    grd.DefineEdgeTags(delegate (double[] _X) {
        double x = _X[0];
        double y = _X[1];

    if(Math.Abs(y - (-Height/2.0)) < 1.0e-6)
        // bottom
        return IncompressibleBcType.Wall.ToString() + "_bottom";

    if(Math.Abs(y - (+Height/2.0)) < 1.0e-6)
        // top
        return IncompressibleBcType.Wall.ToString()+ "_top";


    if(Math.Abs(x - (0.0)) < 1.0e-6)
        // left
        return IncompressibleBcType.Velocity_Inlet.ToString();

    if(Math.Abs(x - (Length)) < 1.0e-6)
        // right
        return IncompressibleBcType.Pressure_Outlet.ToString();

        throw new ArgumentOutOfRangeException();
    });

    return grd;
};

// set boundary conditions
C.AddBoundaryValue(IncompressibleBcType.Velocity_Inlet.ToString(), "VelocityX", new Formula("X => 1 - X[1] * X[1]", TimeDep: false));

// Set Initial Conditions
C.AddInitialValue("VelocityX", new Formula("X => 0.0", TimeDep: false));

// Solver Options
bool transient = false;
if(transient) {
    C.NoOfTimesteps = 10;
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
    C.dtFixed = 0.01;
} else {
    C.TimesteppingMode = AppControl._TimesteppingMode.Steady;
    C.TimeSteppingScheme = BoSSS.Solution.XdgTimestepping.TimeSteppingScheme.ImplicitEuler;
}

In [7]:
AppControlExtensions.Run(C);


MPI rank 0: NullReferenceException :
Object reference not set to an instance of an object.
   at BoSSS.Solution.Application`1.InitCurrentSessionInfo() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 1407
   at BoSSS.Solution.Application`1.SetUpEnvironment() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 1183
   at BoSSS.Solution.XdgTimestepping.ApplicationWithSolver`1.SetUpEnvironment() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution.XdgTimestepping\ApplicationWithSolver.cs:line 273
   at BoSSS.Solution.Application`1.RunSolverModeInternal() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 2074
   at BoSSS.Solution.Application`1.RunSolverMode() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 2033




Unhandled exception: System.NullReferenceException: Object reference not set to an instance of an object.
   at BoSSS.Solution.Application`1.InitCurrentSessionInfo() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 1407
   at BoSSS.Solution.Application`1.SetUpEnvironment() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 1183
   at BoSSS.Solution.XdgTimestepping.ApplicationWithSolver`1.SetUpEnvironment() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution.XdgTimestepping\ApplicationWithSolver.cs:line 273
   at BoSSS.Solution.Application`1.RunSolverModeInternal() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 2074
   at BoSSS.Solution.Application`1.RunSolverMode() in D:\BoSSS-experimental\public\src\L3-solution\BoSSS.Solution\Application.cs:line 2033
   at BoSSS.Application.BoSSSpad.AppControlExtensions.Run(AppControl ctrl) in D:\BoSSS-experimental\public\src\L4-appli